In [5]:
import numpy as np
import xtrack as xt
import json
from helpers_for_imperfections_model import add_correctors, compute_pseudo_inverse, apply_optics_correction
from helpers_from_xutil import add_chroma_knobs, match_tune_chroma

In [6]:
# CHOOSE SEED HERE
seed = 4
# CHOOSE LINE VERSION HERE
line_version = "LCC_105-0-0_z"

In [7]:
# Load reference line
line0 = xt.Line.from_json(f'line_fccee_p_ring_{line_version}.json')
line0.cycle(name_first_element='rf400', inplace=True) # cycle to rf cavity
line0.configure_radiation(model=None, model_beamstrahlung=None) #disable radiation
line0.twiss_default['method'] = '4d' # switch to 4d Twiss method

tw_ref = line0.twiss(coupling_edw_teng=True, matrix_stability_tol=0.20)
for col in ['f1001', 'f1010']:
    tw_ref[col + "c"] = np.real(tw_ref[col])
    tw_ref[col + "s"] = np.imag(tw_ref[col])

Loading line from dict: 100%|██████████| 21288/21288 [00:02<00:00, 7894.91it/s] 


Done loading line from dict.           


In [8]:
# Load orbit-corrected line (cycled to rf cavity, radiation disabled, 4d method)
line = xt.Line.from_json(f'{line_version}_line_01_orbit_corrected_tune_matched_seed{seed}.json')

Loading line from dict: 100%|██████████| 21288/21288 [00:03<00:00, 5360.55it/s] 


Done loading line from dict.           


### Preparing the line

In [9]:
# Add optics correctors
tt = line.get_table() # using the table from the orbit-corrected line

# Filter out normal quads
ttquad = tt.rows[tt.element_type=='Quadrupole']
mask = [quad for quad in ttquad.name if line.element_dict[quad].k1s==0 and abs(line.element_dict[quad].k1)>0]
ttquadno = ttquad.rows[np.isin(ttquad.name, mask)]

# Filter out normal sexts
ttsext = tt.rows[tt.element_type=='Sextupole']
mask = [sext for sext in ttsext.name if line.element_dict[sext].k2s==0 and abs(line.element_dict[sext].k2)>0]
ttsextno = ttsext.rows[np.isin(ttsext.name, mask)]

# Add optics correctors to quads and sexts
add_correctors(line, ttquadno.name, type='normal', order=1, switch_name='on_qno_corrector') # normal optics correctors at powered normal quads
add_correctors(line, ttsextno.name, type='skew', order=1, switch_name='on_qsk_corrector') # skew optics correctors at powered normal sexts

In [10]:
# Define observation points and observables
OBSERVATION_POINTS = json.load(open(f'helper_files_for_RM/observation_points.json', 'r'))
OBSERVABLES_NORMAL = ['mux', 'muy', 'dx']
OBSERVABLES_SKEW = ['f1001s', 'f1001c', 'f1010s', 'f1010c', 'dy']

In [11]:
# CHOOSE NUMBER OF DESIRED ITERATIONS FOR OPTICS CORRECTION
nloops = 5

### Applying optics correction

In [13]:
# Normal correctors

# Load response matrix for normal correctors
corr_type = 'knl1'
RM_normal = np.load(f'helper_files_for_RM/RM_{corr_type}_{"_".join(OBSERVABLES_NORMAL)}.npy')
corr_knob_names_normal = json.load(open(f'helper_files_for_RM/corr_knob_names_{corr_type}_{"_".join(OBSERVABLES_NORMAL)}.json'))

# Invert RM
RM_inv_normal, U, S, VT = compute_pseudo_inverse(RM_normal)

# Apply optics correction
apply_optics_correction(line, RM_inv_normal, corr_knob_names_normal, tw_ref=tw_ref, OBSERVATION_POINTS=OBSERVATION_POINTS, OBSERVABLES=OBSERVABLES_NORMAL, nloops=nloops)
print('Applied optics correction with normal correctors.')

Applied optics correction with normal correctors.


In [14]:
# Skew correctors

# Load response matrix for skew correctors
corr_type = 'ksl1'
RM_skew = np.load(f'helper_files_for_RM/RM_{corr_type}_{"_".join(OBSERVABLES_SKEW)}.npy')
corr_knob_names_skew = json.load(open(f'helper_files_for_RM/corr_knob_names_{corr_type}_{"_".join(OBSERVABLES_SKEW)}.json'))

# Invert RM
RM_inv_skew, U, S, VT = compute_pseudo_inverse(RM_skew)

# Apply optics correction
apply_optics_correction(line, RM_inv_skew, corr_knob_names_skew, tw_ref=tw_ref, OBSERVATION_POINTS=OBSERVATION_POINTS, OBSERVABLES=OBSERVABLES_SKEW, nloops=nloops)
print('Applied optics correction with skew correctors.')

Applied optics correction with skew correctors.


In [15]:
# Tune and chroma matching
match_tune_chroma(line, tw_ref, match_quantities='tune', method='4d')
add_chroma_knobs(line, optics_type='LCC')
match_tune_chroma(line, tw_ref, match_quantities='chroma', method='4d')
print('Tune and chroma matching completed.')

                                             
Optimize - start penalty: 0.2195                            
Matching: model call n. 6 penalty = 1.2989e-05              
Optimize - end penalty:  1.29887e-05                            
Target status:               alty = 1.2989e-05              
id state tag  tol_met       residue   current_val    target_val description                            
0  ON    tune    True   7.82821e-08       194.167       194.167 'qx', val=194.167, tol=1e-05, weight=10
1  ON    tune    True   -1.2965e-06         170.2         170.2 'qy', val=170.2, tol=1e-05, weight=10  
Vary status:                 
id state tag  met name lower_limit   current_val upper_limit val_at_iter_0          step        weight
0  ON    quad OK  kqf2 None           0.00927126 None           0.00927149         1e-08             1
1  ON    quad OK  kqd1 None           -0.0137028 None            -0.013704         1e-08             1
                                             
Optimize 

In [16]:
# Save the optics-corrected line
line.to_json(f'{line_version}_line_00_optics_corrected_seed{seed}_{nloops}loops_4Dmethod_without_radiation.json')
print(f'Optics-corrected line with radiation disabled saved for seed {seed}.')
print(f'Finished procedure for seed {seed} with radiation disabled.')

Optics-corrected line with radiation disabled saved for seed 4.
Finished procedure for seed 4 with radiation disabled.


### Introducing radiation, tapering, 6d Twiss

In [17]:
# Introduce radiation, tapering, and switch to 6D method
line.twiss_default['method'] = '6d' #switch to 6d Twiss method
line.configure_radiation(model='mean', model_beamstrahlung=None) #enable radiation
line.compensate_radiation_energy_loss() #enable tapering

Compensating energy loss.
Share energy loss among cavities (repeat until energy loss is zero)
Energy loss: 34_700_328.648 eV             
Energy loss: 105_624.586 eV             
Energy loss: 321.879 eV             
Energy loss: 0.980 eV             
Energy loss: -53_648.113 eV             
Energy loss: -244.961 eV             
Energy loss: 162.184 eV             
Energy loss: 1.483 eV             

  - Set delta_taper
  - Restore cavity voltage and frequency. Set cavity lag


### Repeat optics correction if desired

In [18]:
# If desired, repeat the optics correction procedure with radiation enabled, but this only makes a marginal difference in the final optics-corrected line

In [19]:
# Introduce radiation, tapering, and switch to 6D method in the reference line
line0.twiss_default['method'] = '6d' # switch to 6d Twiss method
line0.configure_radiation(model='mean', model_beamstrahlung=None) #enable radiation
line0.compensate_radiation_energy_loss() #enable tapering

tw_ref = line0.twiss(coupling_edw_teng=True, matrix_stability_tol=0.20)
for col in ['f1001', 'f1010']:
    tw_ref[col + "c"] = np.real(tw_ref[col])
    tw_ref[col + "s"] = np.imag(tw_ref[col])

Compensating energy loss.
Share energy loss among cavities (repeat until energy loss is zero)
Energy loss: 34_697_589.445 eV             
Energy loss: 105_606.954 eV             
Energy loss: 321.797 eV             
Energy loss: 0.981 eV             
Energy loss: -53_640.691 eV             
Energy loss: -244.903 eV             
Energy loss: 162.147 eV             
Energy loss: 1.483 eV             

  - Set delta_taper
  - Restore cavity voltage and frequency. Set cavity lag


In [20]:
# Apply optics correction for normal and skew correctors
apply_optics_correction(line, RM_inv_normal, corr_knob_names_normal, tw_ref=tw_ref, OBSERVATION_POINTS=OBSERVATION_POINTS, OBSERVABLES=OBSERVABLES_NORMAL, nloops=nloops)
print('Applied optics correction with normal correctors.')

apply_optics_correction(line, RM_inv_skew, corr_knob_names_skew, tw_ref=tw_ref, OBSERVATION_POINTS=OBSERVATION_POINTS, OBSERVABLES=OBSERVABLES_SKEW, nloops=nloops)
print('Applied optics correction with skew correctors.')

# Tune and chroma matching with 6D method
match_tune_chroma(line, tw_ref, match_quantities='tune', method='6d')
add_chroma_knobs(line, optics_type='LCC')
match_tune_chroma(line, tw_ref, match_quantities='chroma', method='6d')
print('Tune and chroma matching completed with 6D method!')

Applied optics correction with normal correctors.
Applied optics correction with skew correctors.
                                             
Optimize - start penalty: 0.03573                           
Matching: model call n. 6 penalty = 5.5859e-07              
Optimize - end penalty:  5.58586e-07                            
Target status:               alty = 5.5859e-07              
id state tag  tol_met       residue   current_val    target_val description                            
0  ON    tune    True  -1.36581e-08       194.167       194.167 'qx', val=194.167, tol=1e-05, weight=10
1  ON    tune    True   5.41631e-08         170.2         170.2 'qy', val=170.2, tol=1e-05, weight=10  
Vary status:                 
id state tag  met name lower_limit   current_val upper_limit val_at_iter_0          step        weight
0  ON    quad OK  kqf2 None           0.00927143 None           0.00927126         1e-08             1
1  ON    quad OK  kqd1 None            -0.013703 None       

In [21]:
# Get final Twiss after correction
tw_final = line.twiss(coupling_edw_teng=True, matrix_stability_tol=0.20)
for col in ['f1001', 'f1010']:
    tw_final[col + "c"] = np.real(tw_final[col])
    tw_final[col + "s"] = np.imag(tw_final[col])

In [22]:
# Save the optics-corrected line with radiation enabled
line.to_json(f'{line_version}_line_00_optics_corrected_seed{seed}_{nloops}loops_6Dmethod_with_radiation.json')
print(f'Optics-corrected line with radiation enabled saved for seed {seed}.')
print(f'Finished procedure for seed {seed} with radiation enabled.')

Optics-corrected line with radiation enabled saved for seed 4.
Finished procedure for seed 4 with radiation enabled.
